## Combine Depression and Precipitation Data

In [14]:
from pathlib import Path

import pandas as pd

def find_mini1_dir():
    cwd = Path.cwd()
    for candidate in [cwd, cwd.parent, cwd / "mini1"]:
        if (candidate / "data" / "raw").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError("Could not locate the mini1 directory")

mini1_dir = find_mini1_dir()
raw_dir = mini1_dir / "data" / "raw"
interim_dir = mini1_dir / "data" / "interim"
interim_dir.mkdir(parents=True, exist_ok=True)

depression_ca = pd.read_csv(raw_dir / "depression_ca.csv", dtype={"countyfips": str})
precipitation_ca = pd.read_csv(raw_dir / "data.csv", comment="#")

In [15]:
depression_for_join = depression_ca.assign(
    county_join=depression_ca["countyname"].str.strip().str.lower()
)

precipitation_for_join = precipitation_ca.assign(
    county_join=(
        precipitation_ca["Name"]
        .str.replace(r"\s+County$", "", regex=True)
        .str.strip()
        .str.lower()
    )
)

In [16]:
depression_counties = set(depression_for_join["county_join"])
precipitation_counties = set(precipitation_for_join["county_join"])

missing_from_precipitation = sorted(depression_counties - precipitation_counties)
missing_from_depression = sorted(precipitation_counties - depression_counties)

if missing_from_precipitation or missing_from_depression:
    raise ValueError(
        "County names did not match. "
        f"Missing from precipitation: {missing_from_precipitation}; "
        f"missing from depression: {missing_from_depression}"
    )

In [17]:
combined_depression_precipitation = (
    depression_for_join
    .merge(
        precipitation_for_join[["county_join", "Value"]],
        on="county_join",
        how="inner",
    )
    [["stateabbr", "countyname", "depression_adjprev", "Value"]]
    .sort_values("countyname")
    .reset_index(drop=True)
)

combined_depression_precipitation["depression_adjprev"] = pd.to_numeric(
    combined_depression_precipitation["depression_adjprev"], errors="coerce"
)
combined_depression_precipitation["Value"] = pd.to_numeric(
    combined_depression_precipitation["Value"], errors="coerce"
)

combined_depression_precipitation.to_csv(
    interim_dir / "depression_precipitation_ca.csv", index=False
)


In [18]:
combined_depression_precipitation = combined_depression_precipitation.drop(columns=["stateabbr"])
combined_depression_precipitation.head(10)

,countyname,depression_adjprev,Value
0,Alameda,18.9,24.39
1,Alpine,23.5,46.86
2,Amador,23.2,41.47
3,Butte,26.0,46.40
4,Calaveras,25.6,40.81
5,Colusa,21.9,28.54
6,Contra Costa,19.7,23.62
7,Del Norte,24.3,73.63
8,El Dorado,23.9,51.73
9,Fresno,21.3,28.95
